In [1]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

# Load Experiment 2 and Experiment 4 artifacts
preprocessor = joblib.load("preprocessing_pipeline.pkl")
best_model = joblib.load("best_smart_campus_model.pkl")
eng_imputer = joblib.load("engineered_features_imputer.pkl")

# Load Experiment 3 feature-store data
df = pd.read_csv("feature_store/retrieved_features.csv")

print("All previous experiment files loaded successfully!")
print("Dataset shape:", df.shape)
print("Best model:", type(best_model).__name__)
print("Preprocessor:", type(preprocessor).__name__)

All previous experiment files loaded successfully!
Dataset shape: (2400, 22)
Best model: LogisticRegression
Preprocessor: ColumnTransformer


In [2]:
original_features = [
    "occupancy", "temperature", "humidity", "co2_level",
    "hvac_status", "lighting_status", "outdoor_temperature",
    "building_id", "room_type"
]

engineered_features = [
    "hour", "day_of_week", "is_weekend",
    "occupancy_per_co2", "thermal_gap",
    "hvac_occupancy_interaction",
    "avg_occupancy", "max_co2", "hvac_usage_rate"
]

all_features = original_features + engineered_features

X = df[all_features].copy()
y = df["high_energy_usage"].astype(int)

print("Input features:", X.shape)
print("Target:", y.shape)
print("\nTarget distribution:")
print(y.value_counts())

Input features: (2400, 18)
Target: (2400,)

Target distribution:
high_energy_usage
0    1554
1     846
Name: count, dtype: int64


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])

Training records: 1920
Testing records: 480


In [8]:
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.sparse import hstack, csr_matrix

class CampusFeatureTransformer(BaseEstimator, TransformerMixin):

    def __init__(
        self,
        preprocessor,
        eng_imputer,
        original_features,
        engineered_features
    ):
        self.preprocessor = preprocessor
        self.eng_imputer = eng_imputer
        self.original_features = original_features
        self.engineered_features = engineered_features

    def fit(self, X, y=None):
        return self

    def transform(self, X):

        original_data = self.preprocessor.transform(
            X[self.original_features]
        )

        engineered_data = self.eng_imputer.transform(
            X[self.engineered_features]
        )

        combined_data = hstack([
            csr_matrix(original_data),
            csr_matrix(engineered_data)
        ]).toarray()

        return combined_data

In [9]:
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.sparse import hstack, csr_matrix

class CampusFeatureTransformer(BaseEstimator, TransformerMixin):

    def __init__(
        self,
        preprocessor,
        eng_imputer,
        original_features,
        engineered_features
    ):
        self.preprocessor = preprocessor
        self.eng_imputer = eng_imputer
        self.original_features = original_features
        self.engineered_features = engineered_features

    def fit(self, X, y=None):
        # Preprocessing and imputer are already fitted
        return self

    def transform(self, X):

        # Select original features
        X_original = X[self.original_features]

        # Select engineered features
        X_engineered = X[self.engineered_features]

        # Apply the fitted preprocessing pipeline
        X_original_processed = self.preprocessor.transform(
            X_original
        )

        # Apply the fitted engineered-feature imputer
        X_engineered_processed = self.eng_imputer.transform(
            X_engineered
        )

        # Combine both feature groups
        X_combined = hstack([
            csr_matrix(X_original_processed),
            csr_matrix(X_engineered_processed)
        ]).tocsr()

        return X_combined

In [10]:
from sklearn.pipeline import Pipeline

feature_transformer = CampusFeatureTransformer(
    preprocessor=preprocessor,
    eng_imputer=eng_imputer,
    original_features=original_features,
    engineered_features=engineered_features
)

campus_pipeline = Pipeline([
    ("feature_processing", feature_transformer),
    ("classifier", best_model)
])

print("End-to-end ML pipeline created successfully!")
print(campus_pipeline)

End-to-end ML pipeline created successfully!
Pipeline(steps=[('feature_processing',
                 CampusFeatureTransformer(eng_imputer=SimpleImputer(strategy='median'),
                                          engineered_features=['hour',
                                                               'day_of_week',
                                                               'is_weekend',
                                                               'occupancy_per_co2',
                                                               'thermal_gap',
                                                               'hvac_occupancy_interaction',
                                                               'avg_occupancy',
                                                               'max_co2',
                                                               'hvac_usage_rate'],
                                          original_features=['occupancy',
                                    

In [11]:
X_check = feature_transformer.transform(X_train)

print("Transformed training shape:", X_check.shape)
print("Model expected features:", best_model.n_features_in_)

Transformed training shape: (1920, 24)
Model expected features: 24


In [12]:
y_pred = campus_pipeline.predict(X_test)

print("First 10 actual values:")
print(y_test.head(10).values)

print("\nFirst 10 predicted values:")
print(y_pred[:10])

First 10 actual values:
[1 0 1 0 1 1 0 1 0 0]

First 10 predicted values:
[1 0 0 0 1 1 0 1 0 0]


In [13]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("End-to-End Pipeline Evaluation")
print("--------------------------------")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

if hasattr(best_model, "predict_proba"):
    y_prob = campus_pipeline.predict_proba(X_test)[:, 1]
    print("\nROC-AUC:", roc_auc_score(y_test, y_prob))

End-to-End Pipeline Evaluation
--------------------------------
Accuracy : 0.8875
Precision: 0.8010471204188482
Recall   : 0.9053254437869822
F1 Score : 0.85

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.88      0.91       311
           1       0.80      0.91      0.85       169

    accuracy                           0.89       480
   macro avg       0.87      0.89      0.88       480
weighted avg       0.89      0.89      0.89       480


Confusion Matrix:
[[273  38]
 [ 16 153]]

ROC-AUC: 0.9660381666317852


In [14]:
sample_input = X_test.iloc[[0]].copy()

prediction = campus_pipeline.predict(sample_input)[0]

print("Sample input:")
display(sample_input)

print("\nPredicted class:", int(prediction))

if int(prediction) == 1:
    print("Result: HIGH ENERGY USAGE")
else:
    print("Result: NORMAL ENERGY USAGE")

Sample input:


,occupancy,temperature,humidity,co2_level,hvac_status,lighting_status,outdoor_temperature,building_id,room_type,hour,day_of_week,is_weekend,occupancy_per_co2,thermal_gap,hvac_occupancy_interaction,avg_occupancy,max_co2,hvac_usage_rate
1046,25,34.86,66.65,742.7,1,1,32.84,B1,Library,14,3,0,0.033661,2.02,25,8.417092,1196.3,0.681122



Predicted class: 1
Result: HIGH ENERGY USAGE


In [15]:
probability = campus_pipeline.predict_proba(sample_input)[0]

print("Normal energy usage probability:", probability[0])
print("High energy usage probability:", probability[1])

Normal energy usage probability: 0.15150606725423887
High energy usage probability: 0.8484939327457611


In [16]:
import joblib
import os

os.makedirs("deployment", exist_ok=True)

joblib.dump(
    best_model,
    "deployment/best_smart_campus_model.pkl"
)

joblib.dump(
    preprocessor,
    "deployment/preprocessing_pipeline.pkl"
)

joblib.dump(
    eng_imputer,
    "deployment/engineered_features_imputer.pkl"
)

print("All deployment artifacts saved successfully!")
print(os.listdir("deployment"))

All deployment artifacts saved successfully!
['best_smart_campus_model.pkl', 'engineered_features_imputer.pkl', 'preprocessing_pipeline.pkl']


In [22]:
import os

print("MLOPS folder:")
print(os.listdir("."))

print("\nDeployment folder:")
if os.path.exists("deployment"):
    print(os.listdir("deployment"))
else:
    print("Deployment folder does not exist yet.")

MLOPS folder:
['.ipynb_checkpoints', 'app.py', 'best_smart_campus_model.pkl', 'deployment', 'engineered_features_imputer.pkl', 'Exp_2.ipynb', 'Exp_3.ipynb', 'Exp_4.ipynb', 'Exp_5.ipynb', 'feature_store', 'mlflow.db', 'mlruns', 'model_tuning_results.csv', 'preprocessing_pipeline.pkl', 'smart_campus_energy.csv']

Deployment folder:
['best_smart_campus_model.pkl', 'engineered_features_imputer.pkl', 'preprocessing_pipeline.pkl']


In [23]:
import requests

url = "http://127.0.0.1:5001/predict"

sample_data = {
    "occupancy": 25,
    "temperature": 28.5,
    "humidity": 60,
    "co2_level": 450,
    "hvac_status": 1,
    "lighting_status": 1,
    "outdoor_temperature": 32,
    "building_id": "B1",
    "room_type": "Lab",

    "hour": 10,
    "day_of_week": 2,
    "is_weekend": 0,
    "occupancy_per_co2": 25 / 450,
    "thermal_gap": 28.5 - 32,
    "hvac_occupancy_interaction": 25 * 1,
    "avg_occupancy": 9.0,
    "max_co2": 1307.1,
    "hvac_usage_rate": 0.65
}

response = requests.post(url, json=sample_data)

print("Status Code:", response.status_code)
print("Prediction Response:")
print(response.json())

Status Code: 200
Prediction Response:
{'prediction': 1, 'probability_high_usage': 0.801089256142452, 'probability_normal': 0.19891074385754803, 'result': 'High Energy Usage'}


In [24]:
normal_sample = sample_data.copy()

normal_sample.update({
    "occupancy": 2,
    "temperature": 24,
    "humidity": 45,
    "co2_level": 350,
    "hvac_status": 0,
    "lighting_status": 0,
    "outdoor_temperature": 25,
    "occupancy_per_co2": 2 / 350,
    "thermal_gap": 24 - 25,
    "hvac_occupancy_interaction": 2 * 0
})

response2 = requests.post(url, json=normal_sample)

print("Status Code:", response2.status_code)
print("Prediction:", response2.json())

Status Code: 200
Prediction: {'prediction': 0, 'probability_high_usage': 1.3776995316624883e-05, 'probability_normal': 0.9999862230046834, 'result': 'Normal Energy Usage'}


In [25]:
import requests

url = "http://127.0.0.1:5001/predict"

# Intentionally incomplete input
invalid_data = {
    "occupancy": 25,
    "temperature": 28.5,
    "humidity": 60
}

response = requests.post(url, json=invalid_data)

print("Test 1: Missing Features")
print("Status Code:", response.status_code)
print("Response:", response.json())

Test 1: Missing Features
Status Code: 400
Response: {'error': 'Missing input features', 'missing_features': ['co2_level', 'hvac_status', 'lighting_status', 'outdoor_temperature', 'building_id', 'room_type', 'hour', 'day_of_week', 'is_weekend', 'occupancy_per_co2', 'thermal_gap', 'hvac_occupancy_interaction', 'avg_occupancy', 'max_co2', 'hvac_usage_rate']}
